## Initialise

In [ ]:
# Initialise all libraries ------------------------------------

# Download mls211 library if not already available
import os
import urllib.request

## Uncomment when using Colab so libraries are imported.
# !pip install camelot-py
# !apt install ghostscript
# !pip install ghostscript
import camelot

url = "https://raw.githubusercontent.com/dche658/mls211/refs/heads/main/mls211.py"
output_path = "mls211.py"

# Check if the file already exists
if not os.path.exists(output_path):
    print(f"Downloading {url}...")
    urllib.request.urlretrieve(url, output_path)
    print("Download complete.")
else:
    print(f"File '{output_path}' already exists. Skipping download.")

# Remainder of imports here
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mls211 import (
    QuadraticFit,
    format_dataframe,
    round_half_up,
)

## Import Google Drive so files can be accessed in Colab
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Submisisons --------------------------------------------
FILE_DIRECTORY = "F:/submissions"
# FILE_DIRECTORY = "/content/drive/MyDrive/submissions" # Colab
# Specify the directory path
directory_path = Path(FILE_DIRECTORY)
# List all files in the submissions folder

# Get all .pdf files (case-insensitive search using rglob/glob)
pdf_files = list(directory_path.glob("*.pdf"))
pdf_files.sort()

# Print the matching files
for index, pdf in enumerate(pdf_files):
    print(index, pdf.name)  # Print just the filename
    # print(pdf)       # Print the full path

## Extract Data

In [ ]:
# Select desired file by index
filename = pdf_files[0]
print(filename)
tables: list[camelot.core.Table] = camelot.read_pdf(filename, flavor="lattice", pages="2")
extracted = []

for tbl in tables:
    df = tbl.df
    extracted.append(df)

# print(extracted)

# Table 1
tbl1_columns = ["ID","TnI","Mean_Abs","Blanked_Abs"]
df = format_dataframe(extracted[0], del_rows=[0], colnames=tbl1_columns)
tbl1 = df.loc[[0,1,2,3,4,5,7,8],:]
tbl1["TnI"] = tbl1["TnI"].str.replace("?","",regex=False)
tbl1["Mean_Abs"] = tbl1["Mean_Abs"].astype(float, errors="ignore")
tbl1["Blanked_Abs"] = tbl1["Blanked_Abs"].astype(float, errors="ignore")
print(tbl1)

In [ ]:
# Table 2
tbl2_columns = ["ID","Cystatin","Abs","Blanked_Abs"]
df = format_dataframe(extracted[1], del_rows=[0], colnames=tbl2_columns)
tbl2 = df.loc[[0,1,2,3,4,5,7,8],:]
tbl2["Cystatin"] = tbl2["Cystatin"].str.replace("?","",regex=False)
tbl2["Cystatin"] = tbl2["Cystatin"].astype(float, errors="ignore")
tbl2["Abs"] = tbl2["Abs"].astype(float, errors="ignore")
tbl2["Blanked_Abs"] = tbl2["Blanked_Abs"].astype(float, errors="ignore")
print(tbl2)

## Data Analysis

### Troponin

In [ ]:
# Troponin
colnames = tbl1_columns

df = tbl1.loc[0:5,:].copy()
df["TnI"] = df["TnI"].astype(float)
#print(df.dtypes)
analyser = QuadraticFit(df)
reg = analyser.fit("TnI","Blanked_Abs")
# print(reg)
r_squared = analyser.r_squared_through_zero(df["TnI"], df["Blanked_Abs"], reg[0],reg[1])
# print(r_squared)

# Chart the calibration curve
x = df["TnI"]
y = df['Blanked_Abs']
x_smooth = np.linspace(x.min(), x.max(), 300)
trendline = reg[0] * (x_smooth**2) + reg[1] * x_smooth
# Create scatter chart
# Create the plot
plt.figure(figsize=(6, 4))

# Plot the scatter points
plt.scatter(x, y, color="black", alpha=0.7, label="Data Points")

# Plot the trendline
plt.plot(
    x_smooth, trendline, color="black", linestyle="-", linewidth=2, label="Trendline"
)

equation_text = (
    f"$y = {reg[0]:.2e}$x^2 + {reg[1]:.2e}x\n$R^2 = {r_squared:.4f}$"
)

# Place text using axes coordinates (transform=ax.transAxes) so it stays fixed
# (0.05, 0.95 is top-left; 0.95, 0.05 is bottom-right)
plt.gca().text(
    0.05,
    0.95,
    equation_text,
    transform=plt.gca().transAxes,
    fontsize=11,
    verticalalignment="top",
    bbox={"boxstyle": "round, pad=0.5", "facecolor": "white", "alpha": 0.8},
)

# 5. Customize the chart
plt.xlabel("Troponin concentration (ng/L)")
plt.ylabel("Absorbance (Units)")
plt.grid(True, linestyle=":", alpha=0.6)


print("Standard Curve")
print(df)
print(f"Slope: {reg[0]:.2e}, Intercept: {reg[1]:.2e}")
print(f"R-squared: {r_squared:.4f}")

# Display the plot
plt.show()


In [ ]:
# Calculate Patient Values
abs = tbl1.loc[7:8,"Blanked_Abs"]
conc = abs.apply(analyser.inv_quadratic_through_zero, args=(reg[0], reg[1]))
conc = round_half_up(conc,"0")
df_conc = pd.concat([abs,conc], axis=1)
df_conc.columns = ["Blanked_Abs","TnI (ng/L)"]
df_conc.index = ["P1","P2"]
print(df_conc)

### Cystatin C

In [ ]:
# Cystatin
colnames = tbl2_columns

df = tbl2.loc[0:5,:]
df["Cystatin"] = df["Cystatin"].astype(float)
#print(df.dtypes)
analyser = QuadraticFit(df)
reg = analyser.fit("Cystatin","Blanked_Abs")
# print(reg)
r_squared = analyser.r_squared_through_zero(df["Cystatin"], df["Blanked_Abs"], reg[0],reg[1])
# print(r_squared)

# Chart the calibration curve
x = df["Cystatin"]
y = df['Blanked_Abs']
x_smooth = np.linspace(x.min(), x.max(), 300)
trendline = reg[0] * (x_smooth**2) + reg[1] * x_smooth
# Create scatter chart
# Create the plot
plt.figure(figsize=(6, 4))

# Plot the scatter points
plt.scatter(x, y, color="black", alpha=0.7, label="Data Points")

# Plot the trendline
plt.plot(
    x_smooth, trendline, color="black", linestyle="-", linewidth=2, label="Trendline"
)

equation_text = (
    f"$y = {reg[0]:.2e}$x^2 + {reg[1]:.2e}x\n$R^2 = {r_squared:.4f}$"
)

# Place text using axes coordinates (transform=ax.transAxes) so it stays fixed
# (0.05, 0.95 is top-left; 0.95, 0.05 is bottom-right)
plt.gca().text(
    0.05,
    0.95,
    equation_text,
    transform=plt.gca().transAxes,
    fontsize=11,
    verticalalignment="top",
    bbox={"boxstyle": "round, pad=0.5", "facecolor": "white", "alpha": 0.8},
)

# 5. Customize the chart
plt.xlabel("Cystatin concentration (mg/L)")
plt.ylabel("Absorbance (Units)")
plt.grid(True, linestyle=":", alpha=0.6)


print("Standard Curve")
print(df)
print(f"Slope: {reg[0]:.2e}, Intercept: {reg[1]:.2e}")
print(f"R-squared: {r_squared:.4f}")

# Display the plot
plt.show()

# Calculate Patient Values
abs = tbl2.loc[7:8,"Blanked_Abs"]
conc = abs.apply(analyser.inv_quadratic_through_zero, args=(reg[0], reg[1]))
conc = round_half_up(conc,"0")
df_conc = pd.concat([abs,conc], axis=1)
df_conc.columns = ["Blanked_Abs","Cystatin (mg/L)"]
df_conc.index = ["P1","P2"]
print(df_conc)